# 05 Evalute Baseline

Evaluate 40 examples across three systems: pretrained `flan-t5-base`, prompt-engineered `flan-t5-base`, and RAG-grounded `flan-t5-base`. The notebook writes metrics, generations, and a markdown comparison report.

## Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.baseline_eval import run_baseline_evaluation

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
pd.set_option('display.max_colwidth', 180)
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/Lenovo/Desktop/sem 6/Gen_AI Project


## Run Baseline Evaluation

This cell runs 40 test examples. It may take several minutes on a MacBook because it generates outputs for three strategies.

In [2]:
metrics_df, aggregate_df, generations_df = run_baseline_evaluation(
    finetune_test_path=DATA_DIR / 'finetune_test.jsonl',
    chroma_dir=DATA_DIR / 'chroma',
    output_dir=OUTPUTS_DIR,
    sample_size=40,
)

print(f'Metric rows: {len(metrics_df)}')
print(f'Generation rows: {len(generations_df)}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Evaluating example 1/40: paper_summary


Evaluating example 2/40: paper_summary


...

Evaluating example 39/40: evidence_based_qa


Evaluating example 40/40: technical_explanation


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Metric rows: 120
Generation rows: 120


## Aggregate Comparison Table

In [3]:
aggregate_df

,model,bleu,rouge1,rouge2,rougeL,bertscore_f1
0,pretrained,0.012575,0.192634,0.078829,0.151496,0.776992
2,rag_system,0.003444,0.155338,0.054878,0.117740,0.762715
1,prompt_engineered,0.003114,0.120365,0.057948,0.104473,0.744704


## Task-Level Metrics

In [4]:
task_metrics = (
    metrics_df.groupby(['model', 'task'])[['bleu', 'rouge1', 'rouge2', 'rougeL', 'bertscore_f1']]
    .mean()
    .reset_index()
    .sort_values(['task', 'model'])
)
task_metrics

,model,task,bleu,rouge1,rouge2,rougeL,bertscore_f1
0,pretrained,comparative_analysis,0.001807,0.159652,0.030968,0.113134,0.748532
6,prompt_engineered,comparative_analysis,0.000442,0.087207,0.016390,0.062308,0.696704
12,rag_system,comparative_analysis,0.000299,0.088045,0.022411,0.071737,0.721679
1,pretrained,evidence_based_qa,0.000169,0.149818,0.048035,0.124383,0.774870
7,prompt_engineered,evidence_based_qa,0.000099,0.108180,0.029550,0.087210,0.748411
13,rag_system,evidence_based_qa,0.001468,0.143408,0.034195,0.104366,0.762449
2,pretrained,literature_review,0.001290,0.173983,0.048089,0.116566,0.769470
8,prompt_engineered,literature_review,0.000085,0.129714,0.057166,0.101992,0.763113
14,rag_system,literature_review,0.000158,0.131648,0.037198,0.085675,0.764175
3,pretrained,paper_summary,0.054543,0.321201,0.211171,0.280474,0.825700


## Output Files

In [5]:
artifact_paths = [
    OUTPUTS_DIR / 'baseline_40_metrics.csv',
    OUTPUTS_DIR / 'baseline_40_generations.csv',
    OUTPUTS_DIR / 'baseline_40_comparison_table.csv',
    OUTPUTS_DIR / 'baseline_40_comparison.md',
]
pd.DataFrame(
    {
        'artifact': [str(path.relative_to(PROJECT_ROOT)) for path in artifact_paths],
        'exists': [path.exists() for path in artifact_paths],
        'size_bytes': [path.stat().st_size if path.exists() else 0 for path in artifact_paths],
    }
)

,artifact,exists,size_bytes
0,outputs/baseline_40_metrics.csv,True,20253
1,outputs/baseline_40_generations.csv,True,96225
2,outputs/baseline_40_comparison_table.csv,True,385
3,outputs/baseline_40_comparison.md,True,7858


## Sample Generations

In [6]:
generations_df[['example_id', 'strategy', 'task', 'topic', 'candidate']].head(12)

,example_id,strategy,task,topic,candidate
0,1,pretrained,paper_summary,LoRA fine-tuning for transformer language models,We propose a novel CoT fine-tuning strategy for French Toxicity Detection using a Dynamic Weighted Loss (DWL) that emphasizes the model's final decision and significantly impro...
1,1,prompt_engineered,paper_summary,LoRA fine-tuning for transformer language models,ToxiFrench: Benchmarking and Enhancing Language Models via CoT Fine-Tuning for French Toxicity Detection
2,1,rag_system,paper_summary,LoRA fine-tuning for transformer language models,We propose a novel Chain-of-Thought (CoT) fine-tuning strategy using a Dynamic Weighted Loss (DWL) that progressively emphasizes the model's final decision and significantly im...
3,2,pretrained,paper_summary,Transformers for scientific document summarization,"We introduce SciWING, an open-source software toolkit which provides access to pre-trained models for scientific document processing tasks, including citation string parsing an..."
4,2,prompt_engineered,paper_summary,Transformers for scientific document summarization,SciWING
5,2,rag_system,paper_summary,Transformers for scientific document summarization,SciWING -- A Software Toolkit for Scientific Document Processing
6,3,pretrained,paper_summary,Semantic search for research paper exploration,Semantic Jira - Semantic Expert Finder in the Bug Tracking Tool Jira
7,3,prompt_engineered,paper_summary,Semantic search for research paper exploration,Semantic Jira
8,3,rag_system,paper_summary,Semantic search for research paper exploration,We propose a semantic expert recommendation system for the Jira bug tracking system.
9,4,pretrained,paper_summary,Hallucination mitigation in retrieval augmented generation,Evaluating the effectiveness and efficiency of late chunking and contextual retrieval for RAG systems.
